# 06. 2026년 8–9월 사후 검증

2026년 7월 기준으로 고정한 **Ridge 목적지×월 예측**을, 프로젝트 종료 후 확보한 8·9월 국제선 출발 여객 실적과 비교한다. 8월은 예측거리 h1, 9월은 h2다. 모델·목적지 유니버스·예측값은 바꾸지 않는다. 이는 모델 선택용 Validation이나 2026년 1–7월 Fixed Test와 별도의 **사후 성능 확인**이다.

실적은 [항공정보포털 노선별 시계열](https://airportal.go.kr/stats/transport/chartDetail5.do)에서 `국제선·출발·유임+환승·전체 운항` 조건으로 받은 2026년 8–9월 자료다. 다운로드 조건과 해시는 `data/raw_followup_2026/source_manifest.json`에 기록했다.

In [1]:
from pathlib import Path
import json, sys
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'scripts'))
import portfolio_data as pdx

raw_root = ROOT / 'data/raw_followup_2026'
manifest = json.loads((raw_root / 'source_manifest.json').read_text(encoding='utf-8'))
files = {item['metric']: raw_root / item['path'] for item in manifest['files']}
for item in manifest['files']:
    assert pdx.file_sha256(raw_root / item['path']) == item['sha256']
assert manifest['query']['last_yearmonth'] == '202608'
assert manifest['query']['this_yearmonth'] == '202609'
assert manifest['query']['di_gubun'] == 'I' and manifest['query']['arvl_type'] == 'D'
print('원자료 해시·조회 조건 확인:', manifest['retrieved_at_kst'])

원자료 해시·조회 조건 확인: 2026-10-11T04:38:42+09:00


## 원자료 점검

여객·운항 파일의 노선×월 키가 일치하는지 확인한다. `전체 합계`는 집계 검산에만 쓰고 목적지 실적에는 더하지 않는다. 목적지 이름은 SQL 뷰와 같은 규칙으로 괄호 안 코드와 공백·구분자를 제거한다.

In [2]:
specs = {
    'passenger': ('여객(명)', 'passengers'),
    'operations': ('운항(편)', 'operations'),
}
frames, totals = {}, {}
for metric, (excel_col, value_col) in specs.items():
    spec = {'columns': ['노선1', '노선2', '년월', excel_col],
            'rename': {'노선1': 'origin_airport_raw', '노선2': 'destination_airport_raw',
                       '년월': 'period_raw', excel_col: value_col}}
    frame = pdx.prepare_frame(files[metric], spec)
    parts = frame['period_raw'].str.extract(r'^(20\d{2})년\s*(0?[1-9]|1[0-2])월$')
    assert parts.notna().all().all()
    frame['period'] = pd.PeriodIndex(parts[0] + '-' + parts[1].str.zfill(2), freq='M')
    total = frame.loc[frame['origin_airport_raw'].eq('전체 합계')].set_index('period')[value_col]
    detail = frame.loc[frame['origin_airport_raw'].ne('전체 합계')].copy()
    assert detail.groupby('period')[value_col].sum().eq(total).all()
    assert not detail.duplicated(['origin_airport_raw', 'destination_airport_raw', 'period']).any()
    frames[metric], totals[metric] = detail, total

route_keys = ['origin_airport_raw', 'destination_airport_raw', 'period']
assert (frames['passenger'][route_keys].sort_values(route_keys).reset_index(drop=True)
        .equals(frames['operations'][route_keys].sort_values(route_keys).reset_index(drop=True)))
passenger = frames['passenger'].copy()
passenger['dest'] = passenger['destination_airport_raw'].map(pdx.normalize_airport_name)
assert passenger['dest'].notna().all()
actual = passenger.groupby(['period', 'dest'], as_index=False)['passengers'].sum()
print('여객·운항 상세 행:', len(passenger), '| 목적지×월 실적:', len(actual))
display(pd.DataFrame({'실제 여객': totals['passenger'], '실제 운항편': totals['operations']}))

여객·운항 상세 행: 605 | 목적지×월 실적: 385


,실제 여객,실제 운항편
period,,
2026-08,4564311,25719
2026-09,4131638,24870


## 고정 예측과 실적 연결

`outputs/forecast_destination_monthly_2026_07.csv`의 **7월 기준 예측을 읽기만** 한다. 169개 목적지×2개월, 338행을 같은 목적지 키에 연결한다. 원자료에 같은 키의 노선이 없으면 해당 목적지의 실제 여객을 0으로 둔다. 새로 등장한 목적지는 예측 대상 169개에 임의로 추가하지 않는다.

In [3]:
forecast_path = ROOT / 'outputs/forecast_destination_monthly_2026_07.csv'
assert pdx.file_sha256(forecast_path) == '32bfe6cfb707e543d61f67f239aa84468824869b8d76c3ecc5e45475625e1c43'
forecast = pd.read_csv(forecast_path, encoding='utf-8-sig')
forecast['period'] = pd.to_datetime(forecast['예측월']).dt.to_period('M')
forecast = forecast[forecast['period'].isin([pd.Period('2026-08'), pd.Period('2026-09')])].copy()
forecast = forecast.rename(columns={'목적지_DB': 'dest', '예측수요': 'pred'})
assert len(forecast) == 338 and forecast.groupby('period')['dest'].nunique().eq(169).all()
assert forecast['예측기준일'].eq('2026-07-01').all()
assert forecast.groupby('period')['horizon'].first().to_dict() == {
    pd.Period('2026-08'): 1, pd.Period('2026-09'): 2}
assert not forecast.duplicated(['period', 'dest']).any()

score = forecast[['period', 'horizon', 'dest', '목적지', 'pred']].merge(
    actual, on=['period', 'dest'], how='left', validate='one_to_one')
missing = score['passengers'].isna()
score['actual'] = score['passengers'].fillna(0)
score['error'] = score['pred'] - score['actual']
score['abs_error'] = score['error'].abs()
coverage = (score.groupby('period')['actual'].sum() / totals['passenger']).mul(100)
print('평가 행:', len(score), '| 원자료에서 같은 이름이 없는 행:', int(missing.sum()))
print('실적 없는 목적지:', ', '.join(sorted(score.loc[missing, '목적지'].unique())))
print('전체 실적 중 모델 대상 169곳의 비중:', ', '.join(
    f'{period}: {value:.2f}%' for period, value in coverage.items()))

평가 행: 338 | 원자료에서 같은 이름이 없는 행: 16
실적 없는 목적지: 나쉬빌, 다카, 달랏, 돈무앙, 마나스, 시카고록퍼드, 알막툼, 프롬펜
전체 실적 중 모델 대상 169곳의 비중: 2026-08: 97.87%, 2026-09: 97.84%


## 예측 오차

**WAPE = 절대오차 합계 ÷ 실제 여객 합계**다. `편향`은 `(예측 합계 − 실제 합계) ÷ 실제 합계`로 계산하므로 음수는 과소예측이다. 두 달 합계 WAPE는 월별 WAPE의 단순 평균이 아니라 338행의 오차·실적 합계로 다시 계산한다.

In [4]:
by_month = score.groupby('period').agg(
    평가행=('dest', 'size'), 실제합계=('actual', 'sum'), 예측합계=('pred', 'sum'),
    절대오차합계=('abs_error', 'sum'), 오차합계=('error', 'sum')).reset_index()
all_months = pd.DataFrame([{'period': '8–9월 합계', '평가행': len(score),
    '실제합계': score['actual'].sum(), '예측합계': score['pred'].sum(),
    '절대오차합계': score['abs_error'].sum(), '오차합계': score['error'].sum()}])
metrics = pd.concat([by_month, all_months], ignore_index=True)
metrics['WAPE'] = metrics['절대오차합계'] / metrics['실제합계']
metrics['편향'] = metrics['오차합계'] / metrics['실제합계']
view = metrics[['period', '평가행', '실제합계', '예측합계', 'WAPE', '편향']].copy()
view['period'] = view['period'].astype(str)
for col in ['실제합계', '예측합계']:
    view[col] = view[col].map(lambda x: f'{x:,.0f}')
for col in ['WAPE', '편향']:
    view[col] = view[col].map(lambda x: f'{x:.2%}')
display(view.rename(columns={'period': '대상월'}))

largest = score.nlargest(5, 'abs_error')[['period', '목적지', 'actual', 'pred', 'error']].copy()
largest['period'] = largest['period'].astype(str)
for col in ['actual', 'pred', 'error']:
    largest[col] = largest[col].map(lambda x: f'{x:+,.0f}' if col == 'error' else f'{x:,.0f}')
print('절대오차가 큰 목적지×월 5행 (오차 = 예측 − 실제)')
display(largest.rename(columns={'period': '대상월', 'actual': '실제', 'pred': '예측', 'error': '오차'}))

,대상월,평가행,실제합계,예측합계,WAPE,편향
0,2026-08,169,"4,466,997","4,159,692",9.15%,-6.88%
1,2026-09,169,"4,042,497","3,836,724",9.65%,-5.09%
2,8–9월 합계,338,"8,509,494","7,996,416",9.39%,-6.03%


절대오차가 큰 목적지×월 5행 (오차 = 예측 − 실제)


,대상월,목적지,실제,예측,오차
312,2026-09,옌지,"42,168",4,"-42,164"
170,2026-09,도쿄 나리타,"313,349","279,118","-34,231"
39,2026-08,옌지,"55,415","25,354","-30,061"
171,2026-09,후쿠오카,"246,312","217,562","-28,750"
8,2026-08,삿포로,"125,163","99,560","-25,603"


## 검토 목적지별 실제 수요와 예측 수요

프로모션 검토 후보 55곳 중 분석에서 먼저 살펴볼 18곳을 두 갈래로 나눈다. **인기·성장 10곳**은 상품·좌석·수익성을 우선 확인할 대상이고, **탐색·방어 8곳**은 각각 소규모 실험 가능성과 수요 감소 원인을 살펴볼 대상이다. 각 차트는 2026년 8·9월 **두 달 합계**를 비교한다. 목적지별 차트의 가로축 범위는 해당 트랙에 맞춰 표시한다.

### 집행 가능성을 우선 확인할 후보 10곳 — 인기·성장

In [5]:
import plotly.graph_objects as go
from reconcile_forecast_candidates import summarize

# 7월 기준 전체 8–12월 예측에서 05의 후보 선정·정렬 규칙을 재사용한다.
candidate_pool = summarize(pd.read_csv(forecast_path, encoding='utf-8-sig'), 'Ridge')
assert len(candidate_pool) == 55
track_rules = {
    '인기': ('예측수요', False),
    '성장': ('예상증가인원', False),
    '탐색': ('예상증가인원', False),
    '방어': ('예상증가인원', True),
}
chosen = pd.concat([
    candidate_pool.loc[candidate_pool['검토트랙'].eq(track)]
    .sort_values([metric, '목적지_DB'], ascending=[ascending, True]).head(5)
    for track, (metric, ascending) in track_rules.items()
], ignore_index=True)
expected_names = {
    '인기': {'오사카 간사이', '도쿄 나리타', '후쿠오카', '타이베이', '상하이 푸둥'},
    '성장': {'가오슝', '지난', '광저우', '다롄', '시애틀'},
    '탐색': {'타이중', '히로시마', '시즈오카'},
    '방어': {'다낭', '방콕', '나트랑 깜라인', '마닐라', '싱가포르'},
}
assert len(chosen) == 18 and chosen['목적지_DB'].nunique() == 18
for track, names in expected_names.items():
    assert set(chosen.loc[chosen['검토트랙'].eq(track), '목적지']) == names
selected_score = score.merge(
    chosen[['목적지_DB', '검토트랙']].rename(columns={'목적지_DB': 'dest'}),
    on='dest', how='inner', validate='many_to_one')
assert len(selected_score) == 36
assert selected_score.groupby('dest')['period'].nunique().eq(2).all()
two_months = selected_score.groupby(['검토트랙', 'dest', '목적지'], as_index=False).agg(
    actual=('actual', 'sum'), pred=('pred', 'sum'))

PLOTLY_TEMPLATE = 'plotly_white'

def plot_track(track, title):
    order = chosen.loc[chosen['검토트랙'].eq(track), '목적지_DB']
    part = two_months.set_index('dest').loc[order].reset_index()
    fig = go.Figure()
    for field, label, color in [
        ('actual', '실제 여객', '#334155'),
        ('pred', '예측 여객', '#4F86B5'),
    ]:
        fig.add_trace(go.Bar(
            x=part[field], y=part['목적지'], name=label, orientation='h',
            marker_color=color, text=part[field].map(lambda x: f'{x:,.0f}'),
            textposition='outside', cliponaxis=False,
            hovertemplate=f'목적지=%{{y}}<br>{label}=%{{x:,.0f}}명<extra></extra>',
        ))
    fig.update_layout(
        title=f'{title}<br><sup>2026년 8–9월 합계 · 2026년 7월 기준 예측</sup>',
        template=PLOTLY_TEMPLATE, barmode='group', height=150 + 65 * len(part),
        xaxis_title='여객 수 (명)', yaxis_title=None,
        legend={'orientation': 'h', 'y': 1.17, 'x': 1, 'xanchor': 'right'},
        margin={'t': 90, 'r': 100, 'b': 55, 'l': 105},
    )
    fig.update_xaxes(tickformat=',', range=[0, part[['actual', 'pred']].max().max() * 1.25])
    fig.update_yaxes(autorange='reversed')
    fig.show(config={'displaylogo': False})

plot_track('인기', '인기 | 집행 가능성 우선 확인')
plot_track('성장', '성장 | 집행 가능성 우선 확인')

### 추가 검토 대상 8곳 — 탐색·방어

탐색은 작은 규모에서 실험 적합성을 살펴볼 목적지다. 방어는 예상 수요 감소의 원인을 가격·좌석 공급·예약 흐름과 함께 점검할 목적지다. 두 트랙 모두 같은 7월 예측과 8·9월 실적을 사용한다.

In [6]:
plot_track('탐색', '탐색 | 실험 적합성 확인')
plot_track('방어', '방어 | 감소 원인 점검')

## 판단

8월 h1 WAPE는 **9.15%**, 9월 h2 WAPE는 **9.65%**, 두 달 합계는 **9.39%** 다. 두 달 모두 실제보다 낮게 예측했으며, 합계 편향은 **−6.03%** 다. 특히 9월 옌지처럼 개별 목적지에서는 큰 오차가 있으므로 전체 WAPE만으로 모든 후보의 예측이 정확하다고 해석하지 않는다.

이 결과는 7월 예측의 **사후 정확도**다. 8·9월 실적을 모델 학습이나 선택에 다시 넣지 않았고, 실제 캠페인의 예약 증가·수익을 검증한 결과도 아니다. 원자료에는 `프롬펜` 예측 키와 다른 `테초국제공항` 실적 키가 있으나, 증빙되지 않은 이름 통합은 하지 않았다. 위 오차는 고정된 목적지 키 기준이다.